# Découverte du fichier (EDA)

In [1]:
# import de la table de fait
import pandas as pd
import numpy as np
commandes = pd.read_csv("/Users/emmanuelfabre/Documents/Professionel/2025.09_Jedha - Data analyst/09_29 - Fullstack data/3_Projets/6.4_Kiosque à Pizza/2025.12.07-Base fictive/commandes_detaillees_generees_v2.csv", header=0)
commandes.head()

,num_commande,kiosque_id,date_commande,source,client_id,type_ligne,id_article,quantite,prix_unitaire_ttc,prix_unitaire_ht,montant_tva,taux_tva,id_remise,moyen_paiement
0,CMD-248750,k01,2022-07-25 18:36:00,CAISSE,NaN,PRODUIT,PIZ069,1,14.5,13.18,1.32,0.1,NaN,Ticket Resto
1,CMD-248701,k01,2022-07-25 20:03:00,CAISSE,c01031,PRODUIT,PIZ071,1,13.9,12.64,1.26,0.1,NaN,Espèces
2,CMD-248701,k01,2022-07-25 20:03:00,CAISSE,c01031,SUPPLEMENT,SUP038,1,0.5,0.45,0.05,0.1,NaN,Carte Bancaire
3,CMD-248767,k01,2022-08-31 11:16:00,CAISSE,c01312,PRODUIT,PIZ002,1,0.0,0.00,0.00,0.1,REM003,Ticket Resto
4,CMD-248801,k01,2022-08-31 13:15:00,CAISSE,c01112,PRODUIT,PIZ099,1,14.5,13.18,1.32,0.1,NaN,Carte Bancaire


In [2]:
description = commandes.describe(include='all')
description

,num_commande,kiosque_id,date_commande,source,client_id,type_ligne,id_article,quantite,prix_unitaire_ttc,prix_unitaire_ht,montant_tva,taux_tva,id_remise,moyen_paiement
count,261492,261492,261492,261492,198025,261492,261492,261492.0,261492.000000,261492.000000,261492.000000,2.614920e+05,16464,261492
unique,123134,4,109030,4,1177,2,225,NaN,NaN,NaN,NaN,NaN,7,4
top,CMD-207355,k01,2024-09-28 22:34:00,CAISSE,c01324,PRODUIT,PIZ059,NaN,NaN,NaN,NaN,NaN,REM005,Carte Bancaire
freq,7,87771,16,211344,9933,228360,2184,NaN,NaN,NaN,NaN,NaN,2592,126782
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,10.727239,9.750905,0.976334,1.000000e-01,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,5.784312,5.259026,0.525295,3.825420e-13,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.000000,0.000000,0.000000,1.000000e-01,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,5.900000,5.360000,0.540000,1.000000e-01,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,13.900000,12.640000,1.260000,1.000000e-01,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,14.500000,13.180000,1.320000,1.000000e-01,NaN,NaN


In [3]:
type_col = commandes.dtypes
type_col

num_commande          object
kiosque_id            object
date_commande         object
source                object
client_id             object
type_ligne            object
id_article            object
quantite               int64
prix_unitaire_ttc    float64
prix_unitaire_ht     float64
montant_tva          float64
taux_tva             float64
id_remise             object
moyen_paiement        object
dtype: object

In [4]:
# Création des colonnes supplémentaires pour l'EDA évolution du CA
commandes['date_commande'] = pd.to_datetime(commandes['date_commande'])
commandes['mois'] = commandes['date_commande'].dt.month
commandes['annee'] = commandes['date_commande'].dt.year
commandes['annee_mois'] = commandes['date_commande'].dt.strftime('%Y-%m')
commandes["jour_semaine"] = commandes['date_commande'].dt.day_name()
commandes['revenu'] = commandes['quantite'] * commandes['prix_unitaire_ttc']


In [5]:
# création de la colonne "periode_analyse" avec un tag du 1er janvier au 30 novembre de chaque année car il manque décembre 2025
# et exclusion du kiosque k03 pour ne conserver que les kiosques "historiques"
cond_annee = commandes['annee'].isin([2023, 2024, 2025])
cond_mois = commandes['mois'].between(1, 11)
cond_kiosque = commandes['kiosque_id'] != 'k03'
commandes_filtrees = cond_annee & cond_mois & cond_kiosque
commandes['periode_analyse'] = commandes_filtrees.astype(int)

# Analyse du CA des clients identifiables vs non identifiables

In [6]:
# 1. Préparation des données avec les filtres
print("=== ÉTAPE 1: PRÉPARATION DES DONNÉES ===")
commandes_filtrees = commandes[
    (commandes['annee'].isin([2023, 2024, 2025])) &
    (commandes['mois'].between(1, 11)) &
    (commandes['kiosque_id'] != 'k03')
].copy()

# 2. Création de la colonne is_identifiable
print("\n=== ÉTAPE 2: SEGMENTATION DES CLIENTS ===")
commandes_filtrees['is_identifiable'] = np.where(
    (commandes_filtrees['client_id'].notna()) &
    (commandes_filtrees['client_id'] != 0) &
    (commandes_filtrees['client_id'] != ''),
    'Clients identifiés',
    'Clients non identifiés'
)

# 3. Calcul des indicateurs BRUTS (sans formatage)
print("\n=== ÉTAPE 3: CALCUL DES INDICATEURS ===")
# Chiffre d'affaires
ca_brut = commandes_filtrees.groupby(['annee', 'is_identifiable'])['revenu'].sum().reset_index()
ca_brut = ca_brut.rename(columns={'revenu': 'ca_total'})

# Nombre de commandes
nb_com_brut = commandes_filtrees.groupby(['annee', 'is_identifiable'])['num_commande'].nunique().reset_index()
nb_com_brut = nb_com_brut.rename(columns={'num_commande': 'nb_commandes'})

# Fusion et calcul du panier moyen
analyse_brut = ca_brut.merge(nb_com_brut, on=['annee', 'is_identifiable'])
analyse_brut['panier_moyen'] = analyse_brut['ca_total'] / analyse_brut['nb_commandes']

# 4. Formatage pour l'affichage (SEULEMENT POUR L'AFFICHAGE)
print("\n=== ÉTAPE 4: FORMATAGE POUR AFFICHAGE ===")
def format_euro(x):
    if pd.isna(x):
        return "0,00 €"
    return f"{x:,.2f} €".replace(",", " ").replace(".", ",")

def format_nombre(x):
    if pd.isna(x):
        return "0"
    return f"{x:,.0f}".replace(",", " ")

# Création d'une copie pour l'affichage
analyse_formattee = analyse_brut.copy()
analyse_formattee['ca_total (€)'] = analyse_formattee['ca_total'].apply(format_euro)
analyse_formattee['nb_commandes'] = analyse_formattee['nb_commandes'].apply(format_nombre)
analyse_formattee['panier_moyen (€)'] = analyse_formattee['panier_moyen'].apply(format_euro)

# 5. Affichage des résultats
print("\n=== ÉTAPE 5: RÉSULTATS FINAUX ===")
print("\nTableau récapitulatif:")
display(analyse_formattee[[
    'annee', 'is_identifiable', 'ca_total (€)', 'nb_commandes', 'panier_moyen (€)'
]].sort_values(['annee', 'is_identifiable']))

# 6. Tableau croisé pour une meilleure comparaison
print("\nTableau croisé dynamique:")
pivot_ca = analyse_formattee.pivot(index='annee', columns='is_identifiable', values='ca_total (€)')
pivot_nb = analyse_formattee.pivot(index='annee', columns='is_identifiable', values='nb_commandes')
pivot_panier = analyse_formattee.pivot(index='annee', columns='is_identifiable', values='panier_moyen (€)')

print("\nChiffre d'affaires:")
display(pivot_ca)
print("\nNombre de commandes:")
display(pivot_nb)
print("\nPanier moyen:")
display(pivot_panier)


=== ÉTAPE 1: PRÉPARATION DES DONNÉES ===

=== ÉTAPE 2: SEGMENTATION DES CLIENTS ===

=== ÉTAPE 3: CALCUL DES INDICATEURS ===

=== ÉTAPE 4: FORMATAGE POUR AFFICHAGE ===

=== ÉTAPE 5: RÉSULTATS FINAUX ===

Tableau récapitulatif:


,annee,is_identifiable,ca_total (€),nb_commandes,panier_moyen (€)
0,2023,Clients identifiés,"576 757,30 €",24 759,"23,29 €"
1,2023,Clients non identifiés,"190 299,65 €",7 571,"25,14 €"
2,2024,Clients identifiés,"617 689,95 €",28 291,"21,83 €"
3,2024,Clients non identifiés,"198 225,95 €",8 497,"23,33 €"
4,2025,Clients identifiés,"557 201,35 €",25 775,"21,62 €"
5,2025,Clients non identifiés,"164 390,00 €",7 142,"23,02 €"



Tableau croisé dynamique:

Chiffre d'affaires:


is_identifiable,Clients identifiés,Clients non identifiés
annee,,
2023,"576 757,30 €","190 299,65 €"
2024,"617 689,95 €","198 225,95 €"
2025,"557 201,35 €","164 390,00 €"



Nombre de commandes:


is_identifiable,Clients identifiés,Clients non identifiés
annee,,
2023,24 759,7 571
2024,28 291,8 497
2025,25 775,7 142



Panier moyen:


is_identifiable,Clients identifiés,Clients non identifiés
annee,,
2023,"23,29 €","25,14 €"
2024,"21,83 €","23,33 €"
2025,"21,62 €","23,02 €"


### Conclusions :
- Entre 2023 et 2025 :
    - baisse des commandes de clients non identifiés avec baisse du panier moyen
    - hausse des commandes de clients identifiables mais baisse du CA car baisse du panier moyen
- le panier moyen des clients non identifiables est plus élevé que celui des client identifiés

# Analyse des Top clients

In [7]:
# 1. Filtrer uniquement les clients identifiables
df_identifiables = commandes_filtrees[commandes_filtrees['is_identifiable'] == 'Clients identifiés']

# 2. Calculer le CA total par client
ca_par_client = df_identifiables.groupby('client_id')['revenu'].sum().sort_values(ascending=False)

# 3. Définir le seuil des 20% (ou le nombre de clients à conserver)
# Top 20% des identifiables
seuil_top_20 = int(len(ca_par_client) * 0.20)
top_clients_ids = ca_par_client.head(seuil_top_20).index.tolist()

# 4. Créer un tag 'is_top_client' dans le DataFrame original
commandes_filtrees['is_top_client'] = commandes_filtrees['client_id'].isin(top_clients_ids)

print(f"Les Top 20% des clients représentent {len(top_clients_ids)} IDs.")

Les Top 20% des clients représentent 226 IDs.


In [8]:
# Filtrer sur les top clients identifiés
df_top_clients = commandes_filtrees[commandes_filtrees['is_top_client'] == True]

evolution_top_clients = df_top_clients.groupby('annee').agg(
    nb_commandes_top=('num_commande', 'nunique'),
    ca_total_top=('revenu', 'sum')
)

evolution_top_clients['panier_moyen_top'] = evolution_top_clients['ca_total_top'] / evolution_top_clients['nb_commandes_top']

print("\n--- Évolution Annuelle du Segment TOP 20% ---")
evolution_top_clients


--- Évolution Annuelle du Segment TOP 20% ---


,nb_commandes_top,ca_total_top,panier_moyen_top
annee,,,
2023,21431,500567.25,23.357158
2024,24470,533997.60,21.822542
2025,22331,482348.10,21.599933


In [9]:
# 1. Calcul des indicateurs pour les top clients
evolution_top_clients = df_top_clients.groupby('annee').agg(
    nb_commandes_top=('num_commande', 'nunique'),
    ca_total_top=('revenu', 'sum')
)

# 2. Calcul du panier moyen
evolution_top_clients['panier_moyen_top'] = evolution_top_clients['ca_total_top'] / evolution_top_clients['nb_commandes_top']

# 3. Fonctions de formatage
def format_euro(x):
    return f"{x:,.2f} €".replace(",", " ").replace(".", ",")

def format_nombre(x):
    return f"{x:,.0f}".replace(",", " ")

def format_pourcentage(x):
    return f"{x:,.1f} %".replace(",", " ").replace(".", ",")

# 4. Ajout de la ligne Total
total_row = pd.DataFrame({
    'nb_commandes_top': [evolution_top_clients['nb_commandes_top'].sum()],
    'ca_total_top': [evolution_top_clients['ca_total_top'].sum()],
    'panier_moyen_top': [evolution_top_clients['ca_total_top'].sum() / evolution_top_clients['nb_commandes_top'].sum()]
}, index=['Total'])

evolution_top_clients = pd.concat([evolution_top_clients, total_row])

# 5. Calcul des variations 2023-2025
if 2023 in evolution_top_clients.index and 2025 in evolution_top_clients.index:
    # Calcul des variations
    variation_nb = (evolution_top_clients.loc[2025, 'nb_commandes_top'] -
                    evolution_top_clients.loc[2023, 'nb_commandes_top']) / \
                   evolution_top_clients.loc[2023, 'nb_commandes_top'] * 100

    variation_ca = (evolution_top_clients.loc[2025, 'ca_total_top'] -
                    evolution_top_clients.loc[2023, 'ca_total_top']) / \
                   evolution_top_clients.loc[2023, 'ca_total_top'] * 100

    variation_panier = (evolution_top_clients.loc[2025, 'panier_moyen_top'] -
                        evolution_top_clients.loc[2023, 'panier_moyen_top']) / \
                       evolution_top_clients.loc[2023, 'panier_moyen_top'] * 100

    # Ajout des variations au DataFrame
    evolution_top_clients.loc['Variation 2023-2025', 'nb_commandes_top'] = format_pourcentage(variation_nb)
    evolution_top_clients.loc['Variation 2023-2025', 'ca_total_top'] = format_pourcentage(variation_ca)
    evolution_top_clients.loc['Variation 2023-2025', 'panier_moyen_top'] = format_pourcentage(variation_panier)

# 6. Formatage des valeurs
evolution_top_clients['nb_commandes_top'] = evolution_top_clients['nb_commandes_top'].apply(
    lambda x: format_nombre(x) if not isinstance(x, str) else x
)
evolution_top_clients['ca_total_top'] = evolution_top_clients['ca_total_top'].apply(
    lambda x: format_euro(x) if not isinstance(x, str) else x
)
evolution_top_clients['panier_moyen_top'] = evolution_top_clients['panier_moyen_top'].apply(
    lambda x: format_euro(x) if not isinstance(x, str) else x
)

# 7. Affichage final
print("\n--- Évolution Annuelle du Segment TOP 20% ---")
display(evolution_top_clients)



--- Évolution Annuelle du Segment TOP 20% ---


/var/folders/bk/9frx0d7n4tb3lvy4_t2cb_xc0000gn/T/ipykernel_49765/2811779801.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4,2 %' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evolution_top_clients.loc['Variation 2023-2025', 'nb_commandes_top'] = format_pourcentage(variation_nb)
/var/folders/bk/9frx0d7n4tb3lvy4_t2cb_xc0000gn/T/ipykernel_49765/2811779801.py:46: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-3,6 %' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  evolution_top_clients.loc['Variation 2023-2025', 'ca_total_top'] = format_pourcentage(variation_ca)
/var/folders/bk/9frx0d7n4tb3lvy4_t2cb_xc0000gn/T/ipykernel_49765/2811779801.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error i

,nb_commandes_top,ca_total_top,panier_moyen_top
2023,21 431,"500 567,25 €","23,36 €"
2024,24 470,"533 997,60 €","21,82 €"
2025,22 331,"482 348,10 €","21,60 €"
Total,68 232,"1 516 912,95 €","22,23 €"
Variation 2023-2025,"4,2 %","-3,6 %","-7,5 %"


### Conclusions :
- 226 clients (20%) constituent environ 80% du CA
- entre 2023 et 2025, le nombre de commandes augmente plus vite que le CA (+13% Vs +6,5%) en raison d'une baisse du panier moyen (-1,35€)

# Analyse des clients fidèles
- Problématique : le gérant constate une baisse de la fréquentation des clients

In [10]:
# 1. Définir les dates d'observation (du 01/04/2023 au 01/12/2025)
# 'MS' signifie 'Month Start' (le 1er du mois)
dates_observation = pd.date_range(start='2023-04-01', end='2025-12-01', freq='MS')

liste_resultats = []

print("Début du traitement par Kiosque...")

for date_obs in dates_observation:
    # 1. Fenêtre glissante [J-90, Date_Obs[
    date_debut = date_obs - pd.Timedelta(days=90)
    
    # 2. Filtrer les commandes de la période
    mask = (commandes['date_commande'] >= date_debut) & (commandes['date_commande'] < date_obs)
    df_window = commandes.loc[mask].copy()
    
    # 3. Calculer le nombre de commandes par Client
    fidelite = df_window.groupby('client_id').size().reset_index(name='nb_commandes_90j')
    
    # Filtrer uniquement les fidèles (>= 3 commandes)
    clients_fideles_ids = fidelite[fidelite['nb_commandes_90j'] >= 3]['client_id']
    
    # Si on a des clients fidèles ce mois-ci...
    if not clients_fideles_ids.empty:
        # On ne garde que les commandes de ces clients fidèles pour déterminer leur kiosque
        commandes_fideles = df_window[df_window['client_id'].isin(clients_fideles_ids)]
        
        # --- 4. DÉTERMINATION DU KIOSQUE DOMINANT ---
        # On trie par date (descendant) pour privilégier le plus récent en cas d'égalité de volume
        commandes_fideles = commandes_fideles.sort_values('date_commande', ascending=False)
        
        # On compte les commandes par Client ET par Kiosque
        kiosque_stats = commandes_fideles.groupby(['client_id', 'kiosque_id']).size().reset_index(name='count')
        
        # On trie pour avoir le kiosque le plus fréquent en premier pour chaque client
        kiosque_stats = kiosque_stats.sort_values(['client_id', 'count'], ascending=[True, False])
        
        # On dédoublonne pour ne garder que la 1ère ligne (le kiosque majoritaire) par client
        kiosque_dominant = kiosque_stats.drop_duplicates(subset=['client_id'], keep='first')[['client_id', 'kiosque_id']]
        
        # --- 5. Finalisation ---
        # On ajoute la date d'observation
        kiosque_dominant['date_observation'] = date_obs
        
        liste_resultats.append(kiosque_dominant)

# --- Consolidation ---
if liste_resultats:
    df_fidelite_kiosque = pd.concat(liste_resultats, ignore_index=True)
    
    print("\n--- Aperçu (Client affecté à son kiosque du moment) ---")
    print(df_fidelite_kiosque.head())


Début du traitement par Kiosque...

--- Aperçu (Client affecté à son kiosque du moment) ---
  client_id kiosque_id date_observation
0    c00016        k04       2023-04-01
1    c00052        k04       2023-04-01
2    c00063        k02       2023-04-01
3    c00068        k04       2023-04-01
4    c00074        k02       2023-04-01


In [11]:
   # --- Analyse Croisée : Évolution par Kiosque ---
evolution_par_kiosque = df_fidelite_kiosque.groupby(['date_observation', 'kiosque_id']).size().unstack(fill_value=0)
    
print("\n--- Évolution des Clients Fidèles par Kiosque ---")
print(evolution_par_kiosque)

evolution_par_kiosque.to_excel("export_clients_fideles.xlsx")


--- Évolution des Clients Fidèles par Kiosque ---
kiosque_id        k01  k02  k03  k04
date_observation                    
2023-04-01        152  183    0  114
2023-05-01        172  162    0  129
2023-06-01        196  159    0  137
2023-07-01        207  145    0  141
2023-08-01        208  144    0  147
2023-09-01        217  137    0  141
2023-10-01        216  140    0  130
2023-11-01        214  145    0  126
2023-12-01        205  160    0  121
2024-01-01        197  167    0  131
2024-02-01        172  174    0  128
2024-03-01        166  188    0  121
2024-04-01        190  186    0  118
2024-05-01        209  169    0  123
2024-06-01        211  165    0  122
2024-07-01        187  182    0  129
2024-08-01        189  190    0  128
2024-09-01        202  180    0  141
2024-10-01        223  164    0  128
2024-11-01        213  179    0  118
2024-12-01        218  162    0  118
2025-01-01        196  157    0  141
2025-02-01        198  148    0  137
2025-03-01        190  1

In [12]:
#Script pour calcul du taux d'attrition

# --- Hypothèses ---
# 1. 'df_fidelite_kiosque' est le résultat du script précédent (colonnes : client_id, kiosque_id, date_observation)
# 2. 'commandes' est votre base de commandes (avec date_commande, client_id)

liste_attrition = []

# On récupère toutes les dates d'observation (les 1er du mois) présentes dans votre étude
dates_uniques = sorted(df_fidelite_kiosque['date_observation'].unique())

print("Calcul de l'attrition en cours...")

for date_debut_mois in dates_uniques:
    # 1. Définir la période de "surveillance" (le mois courant complet)
    # Ex: Si date_debut_mois = 01/04/2023, on regarde jusqu'au 01/05/2023 exclus
    date_fin_mois = date_debut_mois + pd.DateOffset(months=1)
    
    # 2. Isoler les clients fidèles identifiés à ce début de mois
    # On fait une copie pour éviter les avertissements (SettingWithCopyWarning)
    fideles_du_mois = df_fidelite_kiosque[
        df_fidelite_kiosque['date_observation'] == date_debut_mois
    ].copy()
    
    if fideles_du_mois.empty:
        continue

    # 3. Identifier QUI a commandé durant ce mois parmi TOUS les clients
    commandes_du_mois = commandes[
        (commandes['date_commande'] >= date_debut_mois) & 
        (commandes['date_commande'] < date_fin_mois)
    ]
    clients_actifs_ids = set(commandes_du_mois['client_id'].unique())
    
    # 4. Taguer chaque client fidèle : a-t-il commandé ou non ?
    # La colonne 'is_perdu' est Vraie si le client N'EST PAS dans les actifs
    fideles_du_mois['is_perdu'] = ~fideles_du_mois['client_id'].isin(clients_actifs_ids)
    
    # 5. Agréger les résultats par Kiosque
    synthese_mois = fideles_du_mois.groupby(['date_observation', 'kiosque_id']).agg(
        nb_fideles_debut=('client_id', 'count'),
        nb_perdus_fin_mois=('is_perdu', 'sum') # La somme des True donne le nombre de perdus
    ).reset_index()
    
    # Calcul du taux
    synthese_mois['taux_attrition'] = (synthese_mois['nb_perdus_fin_mois'] / synthese_mois['nb_fideles_debut']) * 100
    
    liste_attrition.append(synthese_mois)

# --- Consolidation ---
df_attrition = pd.concat(liste_attrition, ignore_index=True)

# Mise en forme pour l'affichage
pd.options.display.float_format = '{:.2f}%'.format # Format pourcentage
print("\n--- Taux d'Attrition Mensuel des Clients Fidèles par Kiosque ---")
print(df_attrition.head(10))

# --- Optionnel : Vue globale (tous kiosques confondus) ---
df_attrition_global = df_attrition.groupby('date_observation').agg({
    'nb_fideles_debut': 'sum',
    'nb_perdus_fin_mois': 'sum'
}).reset_index()
df_attrition_global['taux_attrition_global'] = (df_attrition_global['nb_perdus_fin_mois'] / df_attrition_global['nb_fideles_debut']) * 100

print("\n--- Évolution Globale du Taux d'Attrition ---")
print(df_attrition_global.head())

df_attrition.to_excel("export_attrition.xlsx")

Calcul de l'attrition en cours...

--- Taux d'Attrition Mensuel des Clients Fidèles par Kiosque ---
  date_observation kiosque_id  nb_fideles_debut  nb_perdus_fin_mois  \
0       2023-04-01        k01               152                  34   
1       2023-04-01        k02               183                  38   
2       2023-04-01        k04               114                  33   
3       2023-05-01        k01               172                  51   
4       2023-05-01        k02               162                  41   
5       2023-05-01        k04               129                  41   
6       2023-06-01        k01               196                  54   
7       2023-06-01        k02               159                  52   
8       2023-06-01        k04               137                  50   
9       2023-07-01        k01               207                  57   

   taux_attrition  
0          22.37%  
1          20.77%  
2          28.95%  
3          29.65%  
4          25.31% 